In [2]:
from __future__ import annotations

import itertools
import math
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple
import numpy as np

# ==============================================================================
# CONFIGURATION SETTINGS (Edit directly in this cell)
# ==============================================================================
CONTCAR_PATH: str = "CONTCAR"          # Input VASP structure file (or POSCAR)
OUTCAR_PATH: str = "OUTCAR"            # Input VASP output file with magnetization
OUTPUT_VESTA_PATH: str = "magmom.vesta"# Output VESTA file

# Atomic Ball Sizing (Shrunk by a factor of 2)
ATOM_RADIUS_SCALE: float = 0.50        # 2x smaller than standard VESTA atom balls
BOND_RADIUS: float = 0.075             # Bond cylinder radius in Å

# Custom Element Color Overrides (RGB 0-255)
# Set any element's color here. Common greys: (165, 165, 165), (144, 144, 144), (128, 128, 128)
CUSTOM_ELEMENT_COLORS: Dict[str, Tuple[int, int, int]] = {
    "Ni": (165, 165, 165),  # Restores Nickel to VESTA's default metallic grey
    "Pd": (165, 165, 165),  # Metallic grey
    "Pt": (165, 165, 165),  # Metallic grey
}

# Magnetization Vector Appearance
MAG_CUTOFF: float = 0.05               # Min |M| (in mu_B) to display an arrow
MAX_ARROW_LENGTH: float = 0.90         # Visual length (in Å) of the largest magnetic moment
ARROW_RADIUS: float = 0.20             # Thickness of vector cylinder (0.20 Å)
COLOR_BY_SPIN_Z: bool = True           # True: Red (+mz / spin-up), Blue (-mz / spin-down)
DEFAULT_COLOR: Tuple[int, int, int] = (255, 40, 40)  # RGB if COLOR_BY_SPIN_Z is False

# Bond Detection Settings
AUTO_GENERATE_BONDS: bool = True       # Automatically detect and draw crystal bonds
BOND_TOLERANCE_FACTOR: float = 1.25    # Cutoff multiplier: d_max = factor * (r_cov1 + r_cov2)
MANUAL_BONDS: Optional[List[Tuple[str, str, float, float]]] = None

# Atom Filtering (Set to None to include all magnetic atoms above cutoff)
TARGET_ELEMENTS: Optional[List[str]] = None  # e.g., ["Fe", "Mn", "Cr"]
TARGET_INDICES: Optional[List[int]] = None   # e.g., [1, 2, 5, 6] (1-based atom indices)
# ==============================================================================


@dataclass
class AtomSite:
    """Represents a single atomic site in the crystal structure."""
    index: int
    species: str
    label: str
    frac_coords: np.ndarray
    raw_moment: np.ndarray = field(default_factory=lambda: np.zeros(3))
    norm_moment: Optional[np.ndarray] = None

    @property
    def clean_species(self) -> str:
        return re.sub(r"[^A-Za-z]", "", self.species)


@dataclass
class ElementProperty:
    """Stores atomic number, baseline covalent/atomic radius, and standard CPK colors."""
    z: int
    radius: float
    rgb: Tuple[int, int, int]
    rgb_diffuse: Tuple[int, int, int] = (204, 204, 204)


@dataclass
class BondSpecification:
    """Represents a bonding criteria rule between two atomic species for VESTA."""
    index: int
    elem1: str
    elem2: str
    d_min: float
    d_max: float
    radius: float = 0.075


class CrystalStructure:
    """Encapsulates unit cell geometry, lattice parameters, and atomic sites."""

    def __init__(self, title: str, lattice_matrix: np.ndarray, sites: List[AtomSite]) -> None:
        self.title = title.strip() or "VASP_Magnetic_Structure"
        self.lattice_matrix = np.array(lattice_matrix, dtype=float)
        self.sites = sites

    @property
    def num_atoms(self) -> int:
        return len(self.sites)

    @property
    def unique_species(self) -> List[str]:
        seen = set()
        unique = []
        for s in self.sites:
            sp = s.clean_species
            if sp not in seen:
                seen.add(sp)
                unique.append(sp)
        return unique

    def get_cell_parameters(self) -> Tuple[float, float, float, float, float, float]:
        """Return (a, b, c, alpha, beta, gamma) in Angstroms and degrees."""
        a_vec, b_vec, c_vec = self.lattice_matrix
        a = float(np.linalg.norm(a_vec))
        b = float(np.linalg.norm(b_vec))
        c = float(np.linalg.norm(c_vec))

        alpha = math.degrees(math.acos(np.clip(np.dot(b_vec, c_vec) / (b * c), -1.0, 1.0)))
        beta = math.degrees(math.acos(np.clip(np.dot(a_vec, c_vec) / (a * c), -1.0, 1.0)))
        gamma = math.degrees(math.acos(np.clip(np.dot(a_vec, b_vec) / (a * b), -1.0, 1.0)))
        return a, b, c, alpha, beta, gamma

    def cartesian_vector_to_vesta(self, cart_vec: np.ndarray) -> np.ndarray:
        """Convert global Cartesian vectors to VESTA crystallographic basis components."""
        a, b, c, _, _, _ = self.get_cell_parameters()
        frac_vec = np.dot(cart_vec, np.linalg.inv(self.lattice_matrix))
        return np.array([frac_vec[0] * a, frac_vec[1] * b, frac_vec[2] * c])


class AtomPropertyManager:
    """Manages atomic radii and colors aligned with VESTA's native palette."""

    ELEMENT_TABLE: Dict[str, Tuple[int, float, Tuple[int, int, int]]] = {
        "H":  (1,  0.46, (255, 255, 255)), "He": (2,  1.22, (217, 255, 255)),
        "Li": (3,  1.57, (204, 128, 255)), "Be": (4,  1.12, (194, 255,   0)),
        "B":  (5,  0.81, (255, 181, 181)), "C":  (6,  0.77, (144, 144, 144)),
        "N":  (7,  0.74, ( 48,  80, 248)), "O":  (8,  0.74, (255,  13,  13)),
        "F":  (9,  0.72, (144, 224,  80)), "Ne": (10, 1.60, (179, 227, 245)),
        "Na": (11, 1.91, (171,  92, 242)), "Mg": (12, 1.60, (138, 255,   0)),
        "Al": (13, 1.43, (191, 166, 166)), "Si": (14, 1.18, (240, 200, 160)),
        "P":  (15, 1.10, (255, 128,   0)), "S":  (16, 1.04, (255, 255,  48)),
        "Cl": (17, 0.99, ( 31, 240,  31)), "Ar": (18, 1.92, (128, 209, 227)),
        "K":  (19, 2.35, (143,  64, 212)), "Ca": (20, 1.97, ( 61, 255,   0)),
        "Sc": (21, 1.64, (230, 230, 230)), "Ti": (22, 1.47, (191, 194, 199)),
        "V":  (23, 1.35, (166, 166, 171)), "Cr": (24, 1.29, (138, 153, 199)),
        "Mn": (25, 1.37, (156, 122, 199)), "Fe": (26, 1.26, (224, 102,  51)),
        "Co": (27, 1.25, (240, 144, 160)), "Ni": (28, 1.25, (165, 165, 165)), # VESTA grey default
        "Cu": (29, 1.28, (200, 115,  51)), "Zn": (30, 1.37, (125, 128, 176)),
        "Ga": (31, 1.53, (194, 143, 143)), "Ge": (32, 1.22, (102, 143, 143)),
        "As": (33, 1.21, (189, 128, 227)), "Se": (34, 1.04, (255, 161,   0)),
        "Br": (35, 1.14, (166,  41,  41)), "Kr": (36, 1.98, ( 92, 184, 209)),
        "Rb": (37, 2.50, (112,  46, 176)), "Sr": (38, 2.15, (  0, 255,   0)),
        "Y":  (39, 1.82, (148, 255, 255)), "Zr": (40, 1.60, (224, 224, 224)),
        "Nb": (41, 1.47, (115, 194, 201)), "Mo": (42, 1.40, ( 84, 181, 181)),
        "Tc": (43, 1.35, ( 59, 158, 158)), "Ru": (44, 1.34, (165, 165, 165)), # VESTA grey default
        "Rh": (45, 1.34, (165, 165, 165)), "Pd": (46, 1.38, (165, 165, 165)), # VESTA grey default
        "Ag": (47, 1.44, (192, 192, 192)), "Cd": (48, 1.52, (255, 217, 143)),
        "In": (49, 1.67, (166, 117, 115)), "Sn": (50, 1.58, (102, 128, 128)),
        "Sb": (51, 1.41, (158,  99, 181)), "Te": (52, 1.37, (212, 122,   0)),
        "I":  (53, 1.33, (148,   0, 148)), "Xe": (54, 2.18, ( 66, 158, 176)),
        "Cs": (55, 2.72, ( 87,  23, 143)), "Ba": (56, 2.24, (  0, 201,   0)),
        "La": (57, 1.88, (112, 212, 255)), "Ce": (58, 1.82, (255, 255, 199)),
        "Pr": (59, 1.82, (217, 255, 199)), "Nd": (60, 1.82, (199, 255, 199)),
        "Sm": (62, 1.81, (143, 255, 199)), "Eu": (63, 2.06, ( 97, 255, 199)),
        "Gd": (64, 1.79, ( 69, 255, 199)), "Tb": (65, 1.77, ( 48, 255, 199)),
        "Dy": (66, 1.77, ( 31, 255, 199)), "Ho": (67, 1.76, (  0, 255, 156)),
        "Er": (68, 1.75, (  0, 230, 117)), "Tm": (69, 1.74, (  0, 212,  82)),
        "Yb": (70, 1.93, (  0, 191,  56)), "Lu": (71, 1.73, (  0, 171,  36)),
        "Hf": (72, 1.59, ( 77, 194, 255)), "Ta": (73, 1.47, ( 77, 166, 255)),
        "W":  (74, 1.41, ( 33, 148, 214)), "Re": (75, 1.37, ( 38, 125, 171)),
        "Os": (76, 1.35, (165, 165, 165)), "Ir": (77, 1.36, (165, 165, 165)),
        "Pt": (78, 1.39, (165, 165, 165)), "Au": (79, 1.44, (255, 209,  36)),
        "Hg": (80, 1.55, (184, 184, 208)), "Tl": (81, 1.71, (166,  84,  77)),
        "Pb": (82, 1.75, ( 87,  89,  97)), "Bi": (83, 1.70, (158,  79, 181)),
        "Th": (90, 1.80, (  0, 186, 255)), "U":  (92, 1.56, (  0, 143, 255)),
    }

    def __init__(
        self,
        scale_factor: float = 0.50,
        custom_colors: Optional[Dict[str, Tuple[int, int, int]]] = None,
    ) -> None:
        self.scale_factor = scale_factor
        self.custom_colors = custom_colors or {}

    def get_property(self, symbol: str) -> ElementProperty:
        clean = re.sub(r"[^A-Za-z]", "", symbol).capitalize()
        z = 1
        radius = 1.30
        rgb = (165, 165, 165)

        if clean in self.ELEMENT_TABLE:
            z, radius, rgb = self.ELEMENT_TABLE[clean]

        # Apply user custom color override if specified
        if clean in self.custom_colors:
            rgb = self.custom_colors[clean]

        return ElementProperty(z=z, radius=radius, rgb=rgb)


class BondGenerator:
    """Generates pairwise bonding rules using standard covalent radii."""

    def __init__(
        self,
        tolerance_factor: float = 1.25,
        bond_radius: float = 0.075,
        manual_bonds: Optional[Sequence[Tuple[str, str, float, float]]] = None,
    ) -> None:
        self.tolerance_factor = tolerance_factor
        self.bond_radius = bond_radius
        self.manual_bonds = manual_bonds
        self.prop_mgr = AtomPropertyManager()

    def generate(self, structure: CrystalStructure) -> List[BondSpecification]:
        specs: List[BondSpecification] = []

        if self.manual_bonds:
            for idx, (e1, e2, d_min, d_max) in enumerate(self.manual_bonds, start=1):
                specs.append(BondSpecification(idx, e1, e2, d_min, d_max, self.bond_radius))
            return specs

        species = structure.unique_species
        bond_idx = 1
        for e1, e2 in itertools.combinations_with_replacement(species, 2):
            r1 = self.prop_mgr.get_property(e1).radius
            r2 = self.prop_mgr.get_property(e2).radius
            d_max = round(float((r1 + r2) * self.tolerance_factor), 3)
            d_min = 0.400

            specs.append(
                BondSpecification(
                    index=bond_idx,
                    elem1=e1,
                    elem2=e2,
                    d_min=d_min,
                    d_max=d_max,
                    radius=self.bond_radius,
                )
            )
            bond_idx += 1

        return specs


class ContcarParser:
    """Parses VASP POSCAR/CONTCAR files."""

    def __init__(self, filepath: Path | str) -> None:
        self.filepath = Path(filepath)

    def parse(self) -> CrystalStructure:
        if not self.filepath.is_file():
            raise FileNotFoundError(f"Structure file '{self.filepath.resolve()}' not found.")

        lines = [line.strip() for line in self.filepath.read_text(encoding="utf-8").splitlines()]
        if len(lines) < 8:
            raise ValueError(f"File '{self.filepath}' is too short to be a valid CONTCAR.")

        title = lines[0]
        scale = float(lines[1].split()[0])

        raw_lattice = np.array(
            [[float(x) for x in lines[i].split()[:3]] for i in range(2, 5)],
            dtype=float,
        )

        if scale < 0:
            target_vol = abs(scale)
            current_vol = abs(np.linalg.det(raw_lattice))
            lattice_matrix = raw_lattice * ((target_vol / current_vol) ** (1.0 / 3.0))
        else:
            lattice_matrix = raw_lattice * scale

        tokens_line5 = lines[5].split()
        if all(token.isdigit() for token in tokens_line5):
            raise ValueError(
                "VASP 4 format detected (missing species symbols on line 6). "
                "Use a VASP 5+ CONTCAR with element names."
            )

        species_list = tokens_line5
        counts = [int(x) for x in lines[6].split()]
        line_idx = 7

        if lines[line_idx].lower().startswith("s"):
            line_idx += 1

        coord_type = lines[line_idx].lower()
        is_cartesian = coord_type.startswith(("c", "k"))
        line_idx += 1

        total_atoms = sum(counts)
        inv_lattice = np.linalg.inv(lattice_matrix)

        expanded_species: List[str] = []
        for sp, count in zip(species_list, counts):
            expanded_species.extend([sp] * count)

        sites: List[AtomSite] = []
        species_counters: Dict[str, int] = {}

        for i in range(total_atoms):
            raw_pos = np.array([float(x) for x in lines[line_idx + i].split()[:3]], dtype=float)
            if is_cartesian:
                cart_pos = raw_pos * (scale if scale > 0 else 1.0)
                frac_pos = np.dot(cart_pos, inv_lattice)
            else:
                frac_pos = raw_pos

            frac_pos = np.mod(frac_pos, 1.0)

            sp = expanded_species[i]
            species_counters[sp] = species_counters.get(sp, 0) + 1
            clean_sp = re.sub(r"[^A-Za-z]", "", sp)
            label = f"{clean_sp}{species_counters[sp]}"

            sites.append(
                AtomSite(
                    index=i + 1,
                    species=sp,
                    label=label,
                    frac_coords=frac_pos,
                )
            )

        return CrystalStructure(title=title, lattice_matrix=lattice_matrix, sites=sites)


class OutcarMagnetizationParser:
    """Parses magnetic moments from the last electronic/ionic step in OUTCAR."""

    def __init__(self, filepath: Path | str, num_atoms: int) -> None:
        self.filepath = Path(filepath)
        self.num_atoms = num_atoms

    def parse(self) -> np.ndarray:
        if not self.filepath.is_file():
            raise FileNotFoundError(f"OUTCAR file '{self.filepath.resolve()}' not found.")

        content = self.filepath.read_text(encoding="utf-8", errors="ignore")
        lines = content.splitlines()

        mag_blocks: Dict[str, List[float]] = {"x": [], "y": [], "z": []}
        i = 0
        n_lines = len(lines)

        while i < n_lines:
            line = lines[i].strip()
            if line.startswith("magnetization (") and line.endswith(")"):
                direction = line.split("(")[1].split(")")[0].strip().lower()
                if direction in mag_blocks:
                    while i < n_lines and not lines[i].strip().startswith("---"):
                        i += 1
                    i += 1

                    step_moments: List[float] = []
                    for _ in range(self.num_atoms):
                        if i >= n_lines:
                            break
                        parts = lines[i].split()
                        if len(parts) >= 2 and parts[0].isdigit():
                            step_moments.append(float(parts[-1]))
                        i += 1

                    if len(step_moments) == self.num_atoms:
                        mag_blocks[direction] = step_moments
            i += 1

        if not mag_blocks["x"]:
            raise ValueError(
                f"No 'magnetization (x)' block matching {self.num_atoms} atoms was found. "
                "Ensure LORBIT = 10 or 11 was set in your INCAR."
            )

        is_noncollinear = bool(mag_blocks["y"] and mag_blocks["z"])
        moments = np.zeros((self.num_atoms, 3), dtype=float)

        if is_noncollinear:
            moments[:, 0] = mag_blocks["x"]
            moments[:, 1] = mag_blocks["y"]
            moments[:, 2] = mag_blocks["z"]
        else:
            moments[:, 2] = mag_blocks["x"]

        return moments


class MagnetizationProcessor:
    """Normalizes magnetic moments proportionally to their actual magnitudes."""

    def __init__(
        self,
        cutoff: float = 0.05,
        max_arrow_length: float = 0.90,
        target_elements: Optional[Sequence[str]] = None,
        target_indices: Optional[Sequence[int]] = None,
    ) -> None:
        self.cutoff = cutoff
        self.max_arrow_length = max_arrow_length
        self.target_elements = {el.lower() for el in target_elements} if target_elements else None
        self.target_indices = set(target_indices) if target_indices else None

    def process(self, structure: CrystalStructure, raw_moments: np.ndarray) -> None:
        magnitudes = np.linalg.norm(raw_moments, axis=1)
        max_mag = float(np.max(magnitudes)) if len(magnitudes) > 0 else 0.0

        if max_mag == 0.0:
            print("Notice: Maximum magnetic moment across all sites is 0.0 mu_B.")
            return

        for site, vec, mag in zip(structure.sites, raw_moments, magnitudes):
            site.raw_moment = vec

            if mag < self.cutoff:
                site.norm_moment = None
                continue

            if self.target_elements is not None and site.clean_species.lower() not in self.target_elements:
                site.norm_moment = None
                continue

            if self.target_indices is not None and site.index not in self.target_indices:
                site.norm_moment = None
                continue

            scaled_cart = (vec / max_mag) * self.max_arrow_length
            site.norm_moment = structure.cartesian_vector_to_vesta(scaled_cart)


class VestaWriter:
    """Writes the crystal structure, SITET/ATOMT radii, bonds, and vectors into .vesta."""

    def __init__(
        self,
        arrow_radius: float = 0.20,
        color_rgb: Tuple[int, int, int] = (255, 40, 40),
        color_by_spin_z: bool = True,
        bond_specs: Optional[List[BondSpecification]] = None,
        atom_radius_scale: float = 0.50,
        custom_colors: Optional[Dict[str, Tuple[int, int, int]]] = None,
    ) -> None:
        self.arrow_radius = arrow_radius
        self.color_rgb = color_rgb
        self.color_by_spin_z = color_by_spin_z
        self.bond_specs = bond_specs or []
        self.atom_radius_scale = atom_radius_scale
        self.prop_mgr = AtomPropertyManager(
            scale_factor=atom_radius_scale,
            custom_colors=custom_colors,
        )

    def write(self, structure: CrystalStructure, output_path: Path | str) -> None:
        out_file = Path(output_path)
        a, b, c, alpha, beta, gamma = structure.get_cell_parameters()
        active_sites = [s for s in structure.sites if s.norm_moment is not None]

        lines: List[str] = [
            "#VESTA_FORMAT_VERSION 3.5.4",
            "",
            "CRYSTAL",
            "",
            "TITLE",
            f"{structure.title}",
            "",
            "GROUP",
            "1 1 P 1",
            "SYMOP",
            " 0.000000 0.000000 0.000000  1  0  0   0  1  0   0  0  1   1",
            " -1.0 -1.0 -1.0  0 0 0  0 0 0  0 0 0",
            "TRANM 0",
            " 0.000000 0.000000 0.000000  1  0  0   0  1  0   0  0  1",
            "LTRANSL",
            " -1",
            " 0.000000 0.000000 0.000000  0.000000 0.000000 0.000000",
            "LORIENT",
            " -1   0   0   0   0",
            " 1.000000  0.000000  0.000000  1.000000  0.000000  0.000000",
            " 0.000000  0.000000  1.000000  0.000000  0.000000  1.000000",
            "LMATRIX",
            " 1.000000  0.000000  0.000000  0.000000",
            " 0.000000  1.000000  0.000000  0.000000",
            " 0.000000  0.000000  1.000000  0.000000",
            " 0.000000  0.000000  0.000000  1.000000",
            " 0.000000  0.000000  0.000000",
            "CELLP",
            f" {a:10.6f} {b:10.6f} {c:10.6f} {alpha:10.6f} {beta:10.6f} {gamma:10.6f}",
            " 0.000000   0.000000   0.000000   0.000000   0.000000   0.000000",
            "STRUC",
        ]

        # Atomic sites block
        for site in structure.sites:
            u, v, w = site.frac_coords
            lines.append(
                f"  {site.index:3d} {site.clean_species:<4s} {site.label:<6s} "
                f"1.0000  {u:10.6f}  {v:10.6f}  {w:10.6f}   1a       1"
            )
            lines.append("    0.000000   0.000000   0.000000  0.00")
        lines.append("  0 0 0 0 0 0 0")

        # SITET block (Per-site radius and RGB)
        lines.append("SITET")
        for site in structure.sites:
            prop = self.prop_mgr.get_property(site.clean_species)
            scaled_r = prop.radius * self.atom_radius_scale
            r, g, b = prop.rgb
            rd, gd, bd = prop.rgb_diffuse
            lines.append(
                f"  {site.index:3d}  {site.label:<6s} {scaled_r:7.4f}  "
                f"{r:3d} {g:3d} {b:3d}  {rd:3d} {gd:3d} {bd:3d}  0"
            )
        lines.append("  0 0")

        # ATOMT block (Per-element defaults)
        lines.append("ATOMT")
        for elem in structure.unique_species:
            prop = self.prop_mgr.get_property(elem)
            scaled_r = prop.radius * self.atom_radius_scale
            r, g, b = prop.rgb
            rd, gd, bd = prop.rgb_diffuse
            lines.append(
                f"  {prop.z:3d} {elem:<4s} {scaled_r:7.4f}  "
                f"{r:3d} {g:3d} {b:3d}  {rd:3d} {gd:3d} {bd:3d}"
            )
        lines.append("  0 0")

        # Bond definitions (SBOND)
        if self.bond_specs:
            lines.append("SBOND")
            for b in self.bond_specs:
                lines.append(
                    f"  {b.index:3d}  {b.elem1:<4s} {b.elem2:<4s} "
                    f"{b.d_min:8.4f} {b.d_max:8.4f}  0  1  1  0  1  {b.radius:6.4f}  2.0000"
                )
            lines.append("  0 0 0 0 0")

        # Vector coordinates (VECTR)
        lines.append("VECTR")
        for vec_idx, site in enumerate(active_sites, start=1):
            vx, vy, vz = site.norm_moment
            lines.append(f"  {vec_idx:4d} {vx:11.6f} {vy:11.6f} {vz:11.6f} 0")
            lines.append(f"  {site.index:4d}    0    0    0    0")
            lines.append("     0    0    0    0    0")
        lines.append("     0    0    0    0    0")

        # Vector appearance styles (VECTT)
        lines.append("VECTT")
        for vec_idx, site in enumerate(active_sites, start=1):
            if self.color_by_spin_z:
                r, g, b_col = (255, 40, 40) if site.raw_moment[2] >= 0.0 else (40, 100, 255)
            else:
                r, g, b_col = self.color_rgb
            lines.append(f"  {vec_idx:4d} {self.arrow_radius:6.3f} {r:3d} {g:3d} {b_col:3d} 1")
        lines.append("     0    0    0    0    0")

        # Visual display mode
        lines.extend([
            "SPLAN",
            "  0 0 0 0",
            "STYLE",
            "DISPF 37753794",
            "MODEL 0 1 0",
            "SURFS 0 1 1",
            "SECTS 32 1",
            "FORMS 0 1",
            "ATOMS 0 0 1",
            "BONDS 1",
            "POLYS 1",
            "VECTS 1.000000",
            "",
        ])

        out_file.write_text("\n".join(lines), encoding="utf-8")


# ==============================================================================
# PIPELINE EXECUTION (Directly runs when executing the cell)
# ==============================================================================
# 1. Parse Structure from CONTCAR
structure = ContcarParser(CONTCAR_PATH).parse()

# 2. Extract final magnetization values from OUTCAR
raw_moments = OutcarMagnetizationParser(OUTCAR_PATH, structure.num_atoms).parse()

# 3. Process vectors proportionally relative to the largest magnetic moment
processor = MagnetizationProcessor(
    cutoff=MAG_CUTOFF,
    max_arrow_length=MAX_ARROW_LENGTH,
    target_elements=TARGET_ELEMENTS,
    target_indices=TARGET_INDICES,
)
processor.process(structure, raw_moments)

# 4. Generate bond connectivity rules (radius = 0.075 Å)
bonds: List[BondSpecification] = []
if AUTO_GENERATE_BONDS:
    bonds = BondGenerator(
        tolerance_factor=BOND_TOLERANCE_FACTOR,
        bond_radius=BOND_RADIUS,
        manual_bonds=MANUAL_BONDS,
    ).generate(structure)

# 5. Write the final .vesta file
writer = VestaWriter(
    arrow_radius=ARROW_RADIUS,
    color_rgb=DEFAULT_COLOR,
    color_by_spin_z=COLOR_BY_SPIN_Z,
    bond_specs=bonds,
    atom_radius_scale=ATOM_RADIUS_SCALE,
    custom_colors=CUSTOM_ELEMENT_COLORS,
)
writer.write(structure, OUTPUT_VESTA_PATH)

# Summary inspection table
active_sites = [s for s in structure.sites if s.norm_moment is not None]
magnitudes = [float(np.linalg.norm(s.raw_moment)) for s in active_sites]
max_mag = max(magnitudes) if magnitudes else 1.0

print(f"Generated '{OUTPUT_VESTA_PATH}' successfully.")
print(f"- Atom balls shrunk by factor of 2 (scale = {ATOM_RADIUS_SCALE*100:.0f}% of default radii):")
for sp in structure.unique_species:
    prop = writer.prop_mgr.get_property(sp)
    print(f"    * {sp}: {prop.radius * ATOM_RADIUS_SCALE:.3f} Å | RGB: {prop.rgb}")
print(f"- Bond thickness (radius): {BOND_RADIUS:.3f} Å")
print(f"- Vector cylinder radius: {ARROW_RADIUS:.2f} Å | Max arrow length: {MAX_ARROW_LENGTH:.2f} Å")
print("-" * 75)
print(f"{'Index':<7} {'Label':<8} {'Raw (mx, my, mz) [mu_B]':<30} {'|M| [mu_B]':<12} {'Arrow Length':<12}")
print("-" * 75)
for s in active_sites:
    mx, my, mz = s.raw_moment
    mag = float(np.linalg.norm(s.raw_moment))
    arrow_len = (mag / max_mag) * MAX_ARROW_LENGTH
    print(f"{s.index:<7d} {s.label:<8s} ({mx:6.3f}, {my:6.3f}, {mz:6.3f})        {mag:6.3f}       {arrow_len:6.3f} Å")

Generated 'magmom.vesta' successfully.
- Atom balls shrunk by factor of 2 (scale = 50% of default radii):
    * Ni: 0.625 Å | RGB: (165, 165, 165)
    * I: 0.665 Å | RGB: (148, 0, 148)
- Bond thickness (radius): 0.075 Å
- Vector cylinder radius: 0.20 Å | Max arrow length: 0.90 Å
---------------------------------------------------------------------------
Index   Label    Raw (mx, my, mz) [mu_B]        |M| [mu_B]   Arrow Length
---------------------------------------------------------------------------
1       Ni1      ( 0.000,  0.000,  1.374)         1.374        0.900 Å
2       Ni2      ( 0.000,  0.000, -1.371)         1.371        0.898 Å
3       Ni3      ( 0.000,  0.000,  1.374)         1.374        0.900 Å
4       I1       ( 0.000,  0.000,  0.069)         0.069        0.045 Å
5       I2       ( 0.000,  0.000,  0.069)         0.069        0.045 Å
6       I3       ( 0.000,  0.000,  0.069)         0.069        0.045 Å
7       I4       ( 0.000,  0.000,  0.069)         0.069        0.045